# Semaine 3 - Pipeline & Optimisation des Modèles

Ce notebook se concentre sur l'automatisation du flux de données via un pipeline scikit-learn complet, la validation robuste par cross-validation, et l'optimisation des hyperparamètres.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import dump, load

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

# Chargement des données
path = "../data/insurance.csv"
df = pd.read_csv(path).drop_duplicates()
print(f"Dataset chargé et nettoyé : {df.shape[0]} lignes")

Dataset chargé et nettoyé : 1337 lignes


## 1. Pipeline Scikit-learn complet

Nous intégrons notre ingénierie de caractéristiques personnalisée dans un objet `BaseEstimator` pour une automatisation totale.

In [2]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_ = X.copy()
        # Interactions et termes polynomiaux
        X_['smoker_bmi'] = X_['smoker'].map({'yes': 1, 'no': 0}) * X_['bmi']
        X_['age_squared'] = X_['age'] ** 2
        return X_

In [3]:
# Définition des colonnes
numeric_features = ["age", "bmi", "children"]
categorical_features = ["sex", "smoker", "region"]
engineered_features = ["age_squared", "smoker_bmi"]

# Preprocessing numérique : Imputation + Scaling
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Preprocessing catégoriel : Imputation + OneHot
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

# Assembleur global
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features + engineered_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Pipeline Maître
master_pipeline = Pipeline(steps=[
    ('engineer', FeatureEngineer()),
    ('preprocessor', preprocessor),
    ('regressor', Ridge())
])

print("Master Pipeline créé avec succès (Ridge par défaut)")

Master Pipeline créé avec succès (Ridge par défaut)


## 2. Validation Robuste (Cross-Validation)

Au lieu d'un simple split, nous utilisons la validation croisée pour évaluer la stabilité du modèle.

In [4]:
X = df.drop("charges", axis=1)
y = df["charges"]

# Split initial pour garder un jeu de test final intact
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_val_score(master_pipeline, X_train, y_train, cv=kf, scoring='r2')

print(f"Moyenne R² (5-Fold CV) : {cv_results.mean():.4f}")
print(f"Écart-type R² : {cv_results.std():.4f}")

Moyenne R² (5-Fold CV) : 0.8212
Écart-type R² : 0.0209


## 3. Optimisation des Hyperparamètres (GridSearch)

Nous allons maintenant tester plusieurs familles de modèles linéaires régularisés (Ridge, Lasso, ElasticNet) et trouver les meilleurs hyperparamètres pour chacun.

In [5]:
# Définition des grilles de paramètres
param_grids = {
    "Ridge": {
        "model": Ridge(),
        "params": {
            "regressor__alpha": [0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0]
        }
    },
    "Lasso": {
        "model": Lasso(max_iter=10000),
        "params": {
            "regressor__alpha": [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
        }
    },
    "ElasticNet": {
        "model": ElasticNet(max_iter=10000),
        "params": {
            "regressor__alpha": [0.001, 0.01, 0.1, 1.0],
            "regressor__l1_ratio": [0.1, 0.5, 0.9]
        }
    }
}

results = []
best_models = {}

print("Démarrage de l'optimisation GridSearchCV...\n")

for name, config in param_grids.items():
    print(f"Optimisation de {name}...")
    
    # Mise à jour du pipeline avec le modèle courant
    pipeline = Pipeline(steps=[
        ('engineer', FeatureEngineer()),
        ('preprocessor', preprocessor),
        ('regressor', config['model'])
    ])
    
    # GridSearch
    grid = GridSearchCV(pipeline, config['params'], cv=5, scoring='r2', n_jobs=-1)
    grid.fit(X_train, y_train)
    
    # Stockage des résultats
    best_score = grid.best_score_
    best_params = grid.best_params_
    best_models[name] = grid.best_estimator_
    
    results.append({
        "Modèle": name,
        "Best R² (CV)": best_score,
        "Best Params": str(best_params)
    })
    
    print(f"  -> Meilleur R² : {best_score:.4f}")
    print(f"  -> Paramètres : {best_params}\n")

# Affichage du tableau récapitulatif
results_df = pd.DataFrame(results).sort_values(by="Best R² (CV)", ascending=False)
results_df

Démarrage de l'optimisation GridSearchCV...

Optimisation de Ridge...


  -> Meilleur R² : 0.8250
  -> Paramètres : {'regressor__alpha': 0.1}

Optimisation de Lasso...


  -> Meilleur R² : 0.8250
  -> Paramètres : {'regressor__alpha': 1.0}

Optimisation de ElasticNet...


  -> Meilleur R² : 0.8250
  -> Paramètres : {'regressor__alpha': 0.001, 'regressor__l1_ratio': 0.9}



,Modèle,Best R² (CV),Best Params
0,Ridge,0.825010,{'regressor__alpha': 0.1}
2,ElasticNet,0.825010,"{'regressor__alpha': 0.001, 'regressor__l1_rat..."
1,Lasso,0.825006,{'regressor__alpha': 1.0}


### Sélection du Meilleur Modèle et Sauvegarde

Nous sélectionnons le modèle ayant obtenu le meilleur score de validation croisée et nous le sauvegardons pour l'application.

In [6]:
best_model_name = results_df.iloc[0]["Modèle"]
final_model = best_models[best_model_name]

print(f"Le modèle vainqueur est : {best_model_name}")

# Évaluation finale sur le jeu de test (jamais vu)
y_pred_final = final_model.predict(X_test)
final_r2 = r2_score(y_test, y_pred_final)
final_mae = mean_absolute_error(y_test, y_pred_final)

print(f"\nPerformance Finale sur Test Set :")
print(f"R²  : {final_r2:.4f}")
print(f"MAE : ${final_mae:.2f}")

# Sauvegarde
model_path = '../models/insurance_model_prod.joblib'
dump(final_model, model_path)
print(f"\nModèle sauvegardé sous : {model_path}")

Le modèle vainqueur est : Ridge

Performance Finale sur Test Set :
R²  : 0.8857
MAE : $2884.67

Modèle sauvegardé sous : ../models/insurance_model_prod.joblib


## 3.1 Analyse des Résidus (Robustesse)

Nous vérifions les hypothèses de la régression linéaire :
1. **Normalité des résidus** (Histogramme + Q-Q Plot)
2. **Homoscédasticité** (Absence de structure dans les résidus vs prédictions)

In [ ]:
import scipy.stats as stats

# Analyse des Résidus
y_pred_test = final_model.predict(X_test)
residuals = y_test - y_pred_test

plt.figure(figsize=(18, 5))

# 1. Residuals vs Predicted
plt.subplot(1, 3, 1)
sns.scatterplot(x=y_pred_test, y=residuals, alpha=0.5)
plt.axhline(0, color='r', linestyle='--')
plt.xlabel('Valeurs Prédites')
plt.ylabel('Résidus')
plt.title('Résidus vs Prédictions (Homoscédasticité)')

# 2. Histogramme
plt.subplot(1, 3, 2)
sns.histplot(residuals, kde=True)
plt.xlabel('Résidus')
plt.title('Distribution des Résidus (Normalité)')

# 3. Q-Q Plot
plt.subplot(1, 3, 3)
stats.probplot(residuals, dist="norm", plot=plt)
plt.title('Q-Q Plot')

plt.tight_layout()
plt.show()

## 4. Interprétabilité du Modèle (SHAP)

Pour comprendre *pourquoi* le modèle fait certaines prédictions, nous utilisons la librairie SHAP (SHapley Additive exPlanations). Cela permet de voir l'impact de chaque variable pour chaque individu.

In [ ]:
import shap

# On doit récupérer l'étape de préprocessing pour transformer les données brutes
preprocessor_step = final_model.named_steps['preprocessor']
feature_engineer_step = final_model.named_steps['engineer']
regressor_step = final_model.named_steps['regressor']

# Transformation des données pour SHAP (car SHAP travaille sur les données transformées)
X_test_transformed = feature_engineer_step.transform(X_test)
X_test_transformed = preprocessor_step.transform(X_test_transformed)

# Récupération des noms de features
feature_names_num = numeric_features + engineered_features
feature_names_cat = preprocessor_step.named_transformers_['cat']['onehot'].get_feature_names_out(categorical_features)
feature_names_all = np.concatenate([feature_names_num, feature_names_cat])

print("Calcul des valeurs SHAP... (LinearExplainer)")
# Utilisation de LinearExplainer car nous avons un modèle linéaire (Ridge/Lasso/ElasticNet)
# masker=X_test_transformed est nécessaire pour les nouvelles versions de SHAP avec des modèles linéaires
explainer = shap.LinearExplainer(regressor_step, X_test_transformed)
shap_values = explainer(X_test_transformed)

# Assignation des noms de features pour les plots
shap_values.feature_names = feature_names_all

print("Valeurs SHAP calculées.")

In [ ]:
plt.figure(figsize=(10, 6))
plt.title("Impact Global des Features (SHAP Summary Plot)")
shap.summary_plot(shap_values, X_test_transformed, feature_names=feature_names_all, show=False)
plt.show()

### Interprétation
- **smoker_yes / smoker_bmi** : Ce sont sans surprise les variables les plus impactantes. Être fumeur (et avoir un BMI élevé en tant que fumeur) augmente drastiquement la prédiction.
- **age / age_squared** : L'âge a un impact significatif et croissant.
- **bmi** : L'impact du BMI seul (pour les non-fumeurs) est modéré mais positif.
- **children** : Avoir des enfants augmente légèrement les coûts.

### Sauvegarde de l'Explainer SHAP

Pour utiliser l'interprétabilité dans l'application Streamlit sans avoir à recharger tout le dataset d'entraînement, nous sauvegardons l'objet `explainer`.

In [ ]:
# Sauvegarde de l'explainer
explainer_path = '../models/shap_explainer.joblib'
dump(explainer, explainer_path)
print(f"Explainer SHAP sauvegardé sous : {explainer_path}")

## 4.2 Feature Importance Globale (Permutation Importance)

Alternative à SHAP, la permutation importance mesure la baisse de performance du modèle lorsqu'on mélange aléatoirement une feature.

In [ ]:
from sklearn.inspection import permutation_importance

print("Calcul de la Permutation Importance...")
result = permutation_importance(
    final_model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1
)

perm_sorted_idx = result.importances_mean.argsort()

plt.figure(figsize=(10, 6))
plt.boxplot(
    result.importances[perm_sorted_idx].T,
    vert=False,
    labels=X_test.columns[perm_sorted_idx]
)
plt.title("Permutation Importance (Test Set)")
plt.tight_layout()
plt.show()